# Sync Metadata Parquet → Delta Lake

Reads `metadata/observations.parquet` from the MinIO landing-zone bucket
and overwrites the Delta Lake table so it matches the Parquet exactly.

Run this cell whenever you want to update the Delta Lake table after ingestion.


In [2]:
#!/usr/bin/env python3
"""
Sync metadata Parquet → Delta Lake.

Reads ``metadata/observations.parquet`` from the MinIO landing-zone bucket
and overwrites the Delta Lake table at
``s3a://<MINIO_BUCKET>/metadata/observations_delta/``
so the table matches the Parquet file exactly.

Run this whenever you want to update the Delta Lake table after ingestion.

Requires:
  - env vars: MINIO_ENDPOINT, MINIO_ACCESS_KEY, MINIO_SECRET_KEY
  - pip install pyarrow deltalake python-dotenv minio

Run:
    python sync_delta.py
"""

import io
import os

from dotenv import load_dotenv
load_dotenv()

import pyarrow as pa
import pyarrow.parquet as pq
from deltalake import write_deltalake
from minio import Minio

# =============================================================================
#  Config
# =============================================================================
MINIO_ENDPOINT = os.environ.get("MINIO_ENDPOINT", "localhost:9000")
MINIO_ACCESS_KEY = os.environ.get("MINIO_ACCESS_KEY", "")
MINIO_SECRET_KEY = os.environ.get("MINIO_SECRET_KEY", "")
MINIO_SECURE = os.environ.get("MINIO_SECURE", "false").lower() == "true"
MINIO_BUCKET = os.environ.get("MINIO_BUCKET", "landing-zone")

PARQUET_KEY = "metadata/observations.parquet"
DELTA_TABLE_PATH = "metadata/observations_delta"


# =============================================================================
#  Helpers
# =============================================================================
def _s3_storage_options():
    scheme = "https" if MINIO_SECURE else "http"
    return {
        "AWS_ACCESS_KEY_ID": MINIO_ACCESS_KEY,
        "AWS_SECRET_ACCESS_KEY": MINIO_SECRET_KEY,
        "AWS_ENDPOINT_URL": f"{scheme}://{MINIO_ENDPOINT}",
        "AWS_REGION": "us-east-1",
        "AWS_S3_ALLOW_UNSAFE_RENAME": "true",
        "AWS_ALLOW_HTTP": "true" if not MINIO_SECURE else "false",
    }


def read_parquet_from_minio(client):
    """Download and read the Parquet metadata file from MinIO."""
    try:
        resp = client.get_object(MINIO_BUCKET, PARQUET_KEY)
        data = resp.read()
        resp.close()
        resp.release_conn()
        return pq.read_table(io.BytesIO(data))
    except Exception as e:
        print(f"  Failed to read {PARQUET_KEY}: {e}")
        return None


def sync_to_delta(table):
    """Overwrite the Delta Lake table with the given PyArrow Table."""
    storage_options = _s3_storage_options()
    delta_uri = f"s3a://{MINIO_BUCKET}/{DELTA_TABLE_PATH}"
    write_deltalake(
        delta_uri,
        table,
        mode="overwrite",
        storage_options=storage_options,
    )
    return delta_uri


# =============================================================================
#  Main
# =============================================================================
def run():
    if not MINIO_ACCESS_KEY or not MINIO_SECRET_KEY:
        raise RuntimeError(
            "MinIO credentials missing. "
            "Set MINIO_ACCESS_KEY and MINIO_SECRET_KEY in your .env file."
        )

    print(f"\n  Connecting to MinIO at {MINIO_ENDPOINT}...")
    client = Minio(
        MINIO_ENDPOINT,
        access_key=MINIO_ACCESS_KEY,
        secret_key=MINIO_SECRET_KEY,
        secure=MINIO_SECURE,
    )

    print(f"  Reading {PARQUET_KEY} from bucket '{MINIO_BUCKET}'...")
    table = read_parquet_from_minio(client)

    if table is None or table.num_rows == 0:
        print("  No Parquet data found — nothing to sync.")
        return

    print(f"  Parquet contains {table.num_rows} rows, {table.num_columns} columns.")
    print(f"  Syncing to Delta Lake...")

    try:
        delta_uri = sync_to_delta(table)
        print(f"\n  Delta Lake synced successfully.")
        print(f"   Table: {delta_uri}")
        print(f"   Rows:  {table.num_rows}\n")
    except Exception as e:
        print(f"\n  Delta Lake sync failed: {e}\n")
        raise


if __name__ == "__main__":
    run()



  Connecting to MinIO at localhost:9000...
  Reading metadata/observations.parquet from bucket 'landing-zone'...
  Parquet contains 12 rows, 38 columns.
  Syncing to Delta Lake...

  Delta Lake synced successfully.
   Table: s3a://landing-zone/metadata/observations_delta
   Rows:  12



In [3]:
# reading the deltalake table

import io
import os

from dotenv import load_dotenv
load_dotenv()

import pyarrow as pa
import pyarrow.parquet as pq
from deltalake import write_deltalake
from minio import Minio

# =============================================================================
#  Config
# =============================================================================
MINIO_ENDPOINT = os.environ.get("MINIO_ENDPOINT", "localhost:9000")
MINIO_ACCESS_KEY = os.environ.get("MINIO_ACCESS_KEY", "")
MINIO_SECRET_KEY = os.environ.get("MINIO_SECRET_KEY", "")
MINIO_SECURE = os.environ.get("MINIO_SECURE", "false").lower() == "true"
MINIO_BUCKET = os.environ.get("MINIO_BUCKET", "landing-zone")

PARQUET_KEY = "metadata/observations.parquet"
DELTA_TABLE_PATH = "metadata/observations_delta"



from deltalake import DeltaTable
from IPython.display import display, HTML
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", 40)

storage_options = _s3_storage_options()
dt = DeltaTable("s3a://landing-zone/metadata/observations_delta", storage_options=storage_options)
df = dt.to_pandas()

print(f"Delta Lake table: {df.shape[0]} rows, {df.shape[1]} columns\n")
display(df)

Delta Lake table: 12 rows, 38 columns



,uuid,time_recorded,processing_version,duration,spectral_centroid_hz,spectral_bandwidth_hz,spectral_rolloff_hz,spectral_flatness,MFCCs,spectral_entropy,zero_crossing_rate,signal_energy,harmonic_energy_ratio,symmetry_score,pattern_stability_score,image_resolution,video_resolution,audio_size,image_size,video_size,processing_time(seconds),kafka_topic,device,audio_format,loudness,source,peak_frequency_hz,peak_time_s,peak_amplitude,peak_rms,audio_path,image_path,video_path,all_peak_frequencies_hz,source_id,tags,description,category
0,cdcc7c1c-5bb0-42c1-a5e3-cf2e8ff5f221,2026-03-30T02:58:03.295086Z,1.1.0,5.0,840.0994088369681,481.5201946965335,1031.0,0.04563443929843885,"[-16.858362, 19.415559, -2.869665, 2...",7.669483255352018,0.030725762928630062,3301.0974550484625,0.06187860045358698,0.9520553450722213,0.9188402501160985,2048x2048,600x600,441044,2581219,2544032,12.355102833000274,-,-,wav,-18.247502480360943,ESC-50,468.0,0.6249433106575963,0.7382641111074456,0.1638537439744318,audio/ESC-50/468/dog_468_cdcc7c1c-5b...,images/ESC-50/468/dog_468_cdcc7c1c-5...,videos/ESC-50/468/dog_468_cdcc7c1c-5...,468|1448|1004|432|952|404|500|328,5-212454-A-0.wav,dog,ESC-50 fold=5 target=0,dog
1,bc431f74-4a6f-40de-b12c-a01c058c0e14,2026-03-30T02:58:12.645561Z,1.1.0,5.0,2112.055673247976,867.664586440742,2739.4,0.1244896207733388,"[-77.779749, 5.984494, -4.890062, 1....",8.125138780486873,0.03903872579920997,3311.98910568878,0.06527225606723962,0.932463163994161,0.9768757470898034,2048x2048,600x600,441044,2920955,1488343,9.263394749999861,-,-,wav,-18.23319694237823,ESC-50,2144.0,0.8749206349206349,0.963496593629166,0.23904744598651992,audio/ESC-50/2144/rooster_2144_bc431...,images/ESC-50/2144/rooster_2144_bc43...,videos/ESC-50/2144/rooster_2144_bc43...,2144|2168|2216|2800|1756|2192|2828|48,1-27724-A-1.wav,rooster,ESC-50 fold=1 target=1,rooster
2,5b6d268c-737d-4379-b68f-1e9abf8a09cd,2026-03-30T02:58:26.003769Z,1.1.0,5.0,1056.7061057863482,1614.240526778635,1282.4,0.28905327120875773,"[11.663231, 8.907922, 0.12106, 2.402...",8.625926686052532,0.09678048426523476,1023.3101510467661,0.04591355458806444,0.9407920831336802,0.9297226449119823,2048x2048,600x600,441044,2662009,2647672,13.403562041999976,-,-,wav,-23.33401311534777,ESC-50,224.0,4.749569160997733,0.9891677518645673,0.13359928657183617,audio/ESC-50/224/hen_224_5b6d268c-73...,images/ESC-50/224/hen_224_5b6d268c-7...,videos/ESC-50/224/hen_224_5b6d268c-7...,896|764|960|844|1024|240|1328|1212,5-244315-B-6.wav,hen,ESC-50 fold=5 target=6,hen
3,f20e7c1c-e6e4-4cc7-bd19-bdefb8b0374d,2026-03-30T02:58:36.833367Z,1.1.0,5.0,112.14212170430574,280.9414307275823,222.0,0.060645470381389906,"[9.155336, 29.124964, -5.225789, 5.8...",5.912747208962019,0.014072626179710565,14887.855274770409,1.532966493675231,0.9298038227558388,0.9287116811964092,2048x2048,600x600,441044,2544126,2437562,10.816464625000663,-,-,wav,-11.705764554486173,ESC-50,4.0,4.624580498866213,0.999969482421875,0.42907374318701413,audio/ESC-50/4/wind_4_f20e7c1c-e6e4-...,images/ESC-50/4/wind_4_f20e7c1c-e6e4...,videos/ESC-50/4/wind_4_f20e7c1c-e6e4...,8|36|68|116|148|320,2-109371-B-16.wav,wind,ESC-50 fold=2 target=16,wind
4,79c6d124-3f7a-49fc-a545-a520d5fab411,2026-03-30T02:58:50.594056Z,1.1.0,5.0,1308.643138077972,2287.669220866893,2781.2000000000003,0.36417936063882256,"[11.385015, 5.163661, -2.942624, 5.6...",6.569321444606143,0.09688025796035356,891.5264831619072,0.5613663877118321,0.9491915307856946,0.9504597696608631,2048x2048,600x600,441044,2524839,2348915,13.693267541999376,-,-,wav,-23.9327434520415,ESC-50,180.0,4.124625850340136,0.3978700745473908,0.11774247012247535,audio/ESC-50/180/insects_180_79c6d12...,images/ESC-50/180/insects_180_79c6d1...,videos/ESC-50/180/insects_180_79c6d1...,360|180|540|2352|4856|3464|4176|1656,5-233787-A-7.wav,insects,ESC-50 fold=5 target=7,insects
5,f938a3c3-16aa-4e20-81dc-838f0d175304,2020-03-01T14:53:40Z,1.1.0,1.723219954648526,3882.0525162706595,1256.8695499675953,5462.448351185621,0.006759229762481775,"[-32.35